# chain-rule-elementwise — faded example 2: Complete abs_back's sign derivative

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `chain-rule-elementwise`. Running the beacon reports progress on the `Backprop: Elementwise chain rule` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Elementwise chain rule` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`chain-rule-elementwise`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "chain-rule-elementwise"
DD_SUBTOPIC = "Backprop: Elementwise chain rule"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The absolute-value op `out = |x|` is elementwise with local derivative `sign(x)` (`+1` for `x>0`, `-1` for `x<0`, and by convention `0` at `x==0`). Backprop is the pointwise product `grad_in = grad_out * sign(x)`. `torch.sign` already encodes the `0`-at-zero convention, so the derivative is just `t.sign(x)`.

## Faded exercise 2

### Faded — abs_back

Implement `abs_back(grad_out, out, x)` for the forward `out = x.abs()`.

The local derivative of `|x|` is `sign(x)`. Complete the blanked line so the returned gradient is `grad_out` scaled position-by-position by the sign of `x`. The shape must equal `x.shape`.

**Fill in:** the elementwise local derivative of abs, namely sign(x)

In [ ]:
def abs_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    local = t.sign(x)
    return grad_out * local


t.manual_seed(0)
x = t.randn(2, 6, dtype=t.float64)
grad_out = t.randn(2, 6, dtype=t.float64)
out = x.abs()
grad_in = abs_back(grad_out, out, x)
print(grad_in.shape)

def _test():
    t.manual_seed(0)
    x = t.randn(2, 6, dtype=t.float64)
    grad_out = t.randn(2, 6, dtype=t.float64)
    out = x.abs()
    got = abs_back(grad_out, out, x)
    xg = x.clone().requires_grad_(True)
    xg.abs().backward(grad_out)
    assert got.shape == x.shape, (got.shape, x.shape)
    assert got.dtype == x.dtype
    assert t.allclose(got, xg.grad, atol=1e-12), (got - xg.grad).abs().max().item()
    # Explicit sign behavior.
    xs = t.tensor([-3.0, 2.0], dtype=t.float64)
    go = t.tensor([1.0, 1.0], dtype=t.float64)
    assert t.allclose(abs_back(go, xs.abs(), xs), t.tensor([-1.0, 1.0], dtype=t.float64))

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def abs_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    local = t.sign(x)
    return grad_out * local


t.manual_seed(0)
x = t.randn(2, 6, dtype=t.float64)
grad_out = t.randn(2, 6, dtype=t.float64)
out = x.abs()
grad_in = abs_back(grad_out, out, x)
print(grad_in.shape)
```
</details>